# 01 — Data Exploration: HHAR & PAMAP2

This notebook explores both datasets used in the GNN+LSTM HAR project:
- **HHAR**: Heterogeneity Human Activity Recognition (phone + watch sensors)
- **PAMAP2**: Physical Activity Monitoring (wrist/chest/ankle IMUs)

Goals:
1. Load raw data and inspect structure
2. Check sampling rates, missing values, class distributions
3. Visualise raw sensor signals per activity

In [ ]:
import sys
sys.path.append('..')  # add project root to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)
print('Libraries loaded.')

## 1. HHAR Dataset

In [ ]:
from src.config import HHAR_RAW_DIR, HHAR_ACTIVITIES
from src.preprocessing import load_hhar_raw

try:
    df_hhar = load_hhar_raw()
    print(f'HHAR shape: {df_hhar.shape}')
    print(f'Columns: {df_hhar.columns.tolist()}')
    df_hhar.head()
except FileNotFoundError as e:
    print(f'[INFO] {e}')
    print('Run: python -m src.data_download  to download data first.')

In [ ]:
# Only run if data loaded successfully
try:
    df_hhar.columns = [c.lower() for c in df_hhar.columns]

    # Activity distribution
    gt_col = 'gt' if 'gt' in df_hhar.columns else 'activity'
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    df_hhar[gt_col].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('HHAR — Activity Distribution')
    axes[0].set_xlabel('Activity')
    axes[0].set_ylabel('Sample Count')

    # User distribution
    df_hhar['user'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('HHAR — Samples per User')
    axes[1].set_xlabel('User')
    axes[1].set_ylabel('Sample Count')
    
    plt.tight_layout()
    plt.show()
except Exception:
    print('Data not loaded — skipping plot.')

In [ ]:
# Visualise raw accelerometer signal for one activity
try:
    activity_to_plot = 'walk'
    sample = df_hhar[df_hhar[gt_col] == activity_to_plot].head(300)

    fig, ax = plt.subplots(figsize=(14, 3))
    for col, color in zip(['x', 'y', 'z'], ['red', 'green', 'blue']):
        if col in sample.columns:
            ax.plot(sample[col].values, label=col, color=color, alpha=0.8)
    ax.set_title(f'HHAR — Raw Accelerometer Signal ({activity_to_plot})')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Acceleration')
    ax.legend()
    plt.tight_layout()
    plt.show()
except Exception:
    print('Data not loaded — skipping plot.')

## 2. PAMAP2 Dataset

In [ ]:
from src.config import PAMAP2_RAW_DIR, PAMAP2_ACTIVITIES
from src.preprocessing import load_pamap2_subject

pamap2_protocol = Path(PAMAP2_RAW_DIR) / 'PAMAP2_Dataset' / 'Protocol'
if not pamap2_protocol.exists():
    pamap2_protocol = Path(PAMAP2_RAW_DIR) / 'Protocol'

try:
    subject_files = sorted(pamap2_protocol.glob('subject*.dat'))
    print(f'Found {len(subject_files)} PAMAP2 subject files.')
    
    # Load first subject
    df_p2 = load_pamap2_subject(subject_files[0])
    print(f'Subject 1 shape: {df_p2.shape}')
    df_p2.head()
except (FileNotFoundError, IndexError, StopIteration) as e:
    print(f'[INFO] PAMAP2 data not found. Run python -m src.data_download first.')

In [ ]:
try:
    # Missing value analysis
    missing_pct = df_p2.isnull().mean() * 100
    top_missing = missing_pct[missing_pct > 0].sort_values(ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Activity distribution
    activity_counts = df_p2['activity_id'].map(PAMAP2_ACTIVITIES).value_counts()
    activity_counts.plot(kind='barh', ax=axes[0], color='teal')
    axes[0].set_title('PAMAP2 — Activity Distribution (Subject 1)')
    axes[0].set_xlabel('Sample Count')
    
    # Missing values
    if len(top_missing) > 0:
        top_missing.head(20).plot(kind='bar', ax=axes[1], color='salmon')
        axes[1].set_title('PAMAP2 — Missing Value % (top 20 columns)')
        axes[1].set_ylabel('% Missing')
    else:
        axes[1].text(0.5, 0.5, 'No missing values', ha='center', va='center')
        axes[1].set_title('PAMAP2 — Missing Values')
    
    plt.tight_layout()
    plt.show()
except Exception:
    print('Data not loaded — skipping plot.')

## 3. Processed Data Summary (after preprocessing)

In [ ]:
from src.config import PROCESSED_DIR

processed_files = list(Path(PROCESSED_DIR).glob('*.npy'))
if processed_files:
    for f in sorted(processed_files):
        arr = np.load(f, allow_pickle=True)
        print(f'{f.name:40s} shape={arr.shape}  dtype={arr.dtype}')
else:
    print('No processed files yet. Run src/preprocessing.py first.')

In [ ]:
# Visualise a few windows from processed data
try:
    X = np.load(f'{PROCESSED_DIR}/pamap2_X.npy')
    y = np.load(f'{PROCESSED_DIR}/pamap2_y.npy')
    
    print(f'PAMAP2 processed: X={X.shape}, y={y.shape}')
    print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')
    
    # Plot 3 random windows
    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    rng = np.random.default_rng(42)
    indices = rng.choice(len(X), 3, replace=False)
    
    for ax, idx in zip(axes, indices):
        ax.plot(X[idx, :, :3])  # plot first 3 channels
        ax.set_title(f'Window {idx} — Class {y[idx]}')
        ax.set_xlabel('Time Step')
    
    plt.suptitle('PAMAP2 — Sample Windows (first 3 channels)', y=1.02)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Processed data not found — run src/preprocessing.py first.')